In [1]:
import json
from datetime import datetime, timezone
import pandas as pd
import csv

In [2]:
def iter_posts(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

In [3]:
def in_range(created_utc, start_utc=None, end_utc=None):
    if created_utc is None:
        return False
    if start_utc is not None and created_utc < start_utc:
        return False
    if end_utc is not None and created_utc >= end_utc:
        return False
    return True

In [4]:
pre_cov_start_date = 1551398400
cov_end_date = 1614556800

In [5]:
import os
os.getcwd()

'/home/benh/NLP_Code/Clean/Pushshift_data_processing'

In [6]:
INPUT_PATH = "../../socialanxiety_submissions"      # JSONL file
OUTPUT_PATH = "../Corpus_data/filtered_posts.csv"

# Only keep these columns (in this exact order)
FIELDS = ['created_utc', "id", "title", "selftext", "author", "url"]
START_UTC = pre_cov_start_date
END_UTC = cov_end_date
with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(out_f, fieldnames=FIELDS, extrasaction="ignore")
    writer.writeheader()

    for post in iter_posts(INPUT_PATH):
        created = post.get("created_utc")
        created = int(float(created))
        if not in_range(created, START_UTC, END_UTC):
            continue
        row = {k: post.get(k, None) for k in FIELDS}
        writer.writerow(row)

In [7]:
posts = pd.read_csv('../Corpus_data/filtered_posts.csv')

In [8]:
posts['Date'] = posts['created_utc'].apply(datetime.utcfromtimestamp)

In [9]:
posts['created_utc'] = posts['Date']

In [10]:
posts = posts[['created_utc', 'id', 'title', 'selftext', 'author', 'url']]

In [11]:
posts = posts.dropna(subset=['selftext'])

In [12]:
posts = posts[posts['selftext'] != '[removed]']

In [13]:
posts = posts[posts['selftext'] != '[deleted]']

In [14]:
posts = posts.reset_index(drop=True)

In [15]:
posts.to_csv('../Corpus_data/filtered_posts.csv',index=False)

Preprocessing the Comments:

In [16]:
INPUT_PATH = "../../socialanxiety_comments"
OUTPUT_PATH = "../Corpus_data/filtered_comments.csv"

FIELDS = ['created_utc', "link_id", "parent_id", "author", "body"]
START_UTC = pre_cov_start_date
END_UTC = cov_end_date
written = 0

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.DictWriter(out_f, fieldnames=FIELDS, extrasaction="ignore")
    writer.writeheader()

    for post in iter_posts(INPUT_PATH):
        created = int(post.get("created_utc"))
        
        if not in_range(created, START_UTC, END_UTC):
            continue
            
        row = {k: post.get(k, None) for k in FIELDS}
        writer.writerow(row)

In [17]:
comments = pd.read_csv('../Corpus_data/filtered_comments.csv')

In [18]:
comments.head()

,created_utc,link_id,parent_id,author,body
0,1551398641,t3_avt3af,t3_avt3af,brucer365,Heartrate increases like I'm about to storm fu...
1,1551398706,t3_avt3af,t1_ehhut3j,brucer365,You know that guy is subbed to this sub
2,1551398797,t3_avpw75,t3_avpw75,brucer365,What I always remember is that people are so s...
3,1551398998,t3_avvxci,t3_avvxci,brucer365,Ever have that teacher that immediately puts Y...
4,1551399358,t3_avoz8l,t1_ehh166c,pasta_girl,What I’ve sensed myself. I guess it’s nothing ...


In [19]:
comments['created_utc'] = pd.to_datetime(comments['created_utc'], unit='s', utc=True)

In [20]:
comments = comments[comments['body'] != '[deleted]']

In [21]:
comments = comments[comments['body'] != '[removed]']

In [22]:
comments = comments.dropna(subset=['body'])

In [23]:
comments = comments.reset_index(drop=True)

In [24]:
comments['link_id'] = comments['link_id'].str[3:]

In [36]:
comments.to_csv('../Corpus_data/filtered_comments.csv',index=False)

In [26]:
comments_grouped = (
    comments
    .groupby("link_id")["body"]
    .apply(lambda texts: "\n\n".join(texts.dropna()))
    .reset_index()
)

In [29]:
docs = posts[['created_utc', 'id', 'title', 'selftext']]

In [30]:
docs['postdoc'] = docs['title'] + '\n\n' + docs['selftext']

/tmp/ipykernel_6235/1688130017.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  docs['postdoc'] = docs['title'] + '\n\n' + docs['selftext']


In [31]:
docs = docs.merge(
    comments_grouped,
    left_on="id",
    right_on="link_id",
    how="left"
)

In [32]:
docs["postdoc"] = (
    docs["postdoc"].fillna("") + " " +
    docs["body"].fillna("")
)

In [33]:
cleandocs = docs[['created_utc', 'id', 'postdoc']]

In [34]:
import re
import html

URL_RE = re.compile(r"""(?xi)
\b(
  https?://[^\s<>"'\]]+ |
  www\.[^\s<>"'\]]+
)\b
""")

MD_LINK_RE = re.compile(r"""\[([^\]]+)\]\((https?://[^)]+)\)""")
ZW_CHARS_RE = re.compile(r"[\u200B\u200C\u200D\u2060\uFEFF]")

# Common ASCII emoticons
EMOTICON_RE = re.compile(r"""
(?:
  [:;=8]                # eyes
  [\-o\*']?             # optional nose
  [\)\]\(\[dDpP/\:\}\{@\|\\]   # mouth
)
""", re.VERBOSE)

# Emoji removal (unicode ranges)
EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "]+",
    flags=re.UNICODE
)

def clean_reddit_text(text):
    if not isinstance(text, str):
        return ""

    t = text

    # HTML unescape (twice handles &amp;#x200B;)
    t = html.unescape(t)
    t = html.unescape(t)

    # Remove zero-width junk
    t = ZW_CHARS_RE.sub("", t)

    # Remove markdown links (keep visible text)
    t = MD_LINK_RE.sub(r"\1", t)

    # Remove URLs
    t = URL_RE.sub("", t)

    # Remove asterisks (bullets/italics markers)
    t = t.replace("*", "")

    # Remove emojis
    t = EMOJI_RE.sub("", t)

    # Remove emoticons like :D :) :(
    t = EMOTICON_RE.sub("", t)

    # Normalize whitespace
    t = re.sub(r"\s+", " ", t).strip()

    return t

In [35]:
cleandocs['postdoc'] = cleandocs['postdoc'].apply(clean_reddit_text)

/tmp/ipykernel_6235/958260616.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleandocs['postdoc'] = cleandocs['postdoc'].apply(clean_reddit_text)


In [37]:
cleandocs.to_csv('../Corpus_data/sa_corpus.csv',index=False)